# 03 - Build Training Datasets

## Stage 4 Objective

Read `data/processed/clauses.csv` and generate two training-ready CSV files:

- `data/processed/simplification_dataset.csv`: a manual-fill template for clause simplification.
- `data/processed/classification_dataset.csv`: weakly labeled clause classification data using keyword rules.

## Labeling Notes

Classification labels are weak labels, not ground truth. They are useful for bootstrapping a small prototype but should be reviewed before model training or reporting.

## 1. Configure Paths and Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_builder import (
    build_classification_dataset,
    build_simplification_dataset,
    class_distribution,
    find_missing_values,
    get_classification_columns,
    get_simplification_columns,
    split_distribution,
)

CLAUSES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'clauses.csv'
SIMPLIFICATION_PATH = PROJECT_ROOT / 'data' / 'processed' / 'simplification_dataset.csv'
CLASSIFICATION_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classification_dataset.csv'

print(f'Clauses path: {CLAUSES_PATH}')
print(f'Simplification output: {SIMPLIFICATION_PATH}')
print(f'Classification output: {CLASSIFICATION_PATH}')

## 2. Load Clause Records

In [ ]:
required_clause_columns = ['clause_id', 'document_id', 'source_path', 'clause_number', 'clause_text']

if CLAUSES_PATH.exists():
    clauses_df = pd.read_csv(CLAUSES_PATH)
else:
    clauses_df = pd.DataFrame(columns=required_clause_columns)

for column in required_clause_columns:
    if column not in clauses_df.columns:
        clauses_df[column] = ''

clauses_df['clause_text'] = clauses_df['clause_text'].fillna('').astype(str)
clauses_df = clauses_df[clauses_df['clause_text'].str.strip() != ''].copy()

print(f'Loaded {len(clauses_df)} usable clause row(s).')
display(clauses_df.head())

## 3. Build Simplification Template

`simple_clause` is intentionally blank. Fill it manually with a faithful plain-language version before Stage 5 model training.

In [ ]:
clause_records = clauses_df.to_dict(orient='records')
simplification_rows = build_simplification_dataset(clause_records, seed=42)
simplification_df = pd.DataFrame(simplification_rows, columns=get_simplification_columns())

print(f'Created {len(simplification_df)} simplification row(s).')
display(simplification_df.head(10))

## 4. Build Weakly Labeled Classification Dataset

Keyword rules assign `clause_type`, `risk_level`, and `risk_type`. The `weak_label_reason` column records which rule fired.

In [ ]:
classification_rows = build_classification_dataset(clause_records, seed=42)
classification_df = pd.DataFrame(classification_rows, columns=get_classification_columns())

print(f'Created {len(classification_df)} classification row(s).')
display(classification_df.head(10))

## 5. Check Missing Labels

In [ ]:
simplification_missing = find_missing_values(
    simplification_rows,
    ['clause_id', 'clause_text', 'simple_clause', 'split'],
)
classification_missing = find_missing_values(
    classification_rows,
    ['clause_id', 'clause_text', 'clause_type', 'risk_level', 'risk_type', 'split'],
)

print('Simplification missing values:')
display(pd.DataFrame([simplification_missing]))
print('Classification missing values:')
display(pd.DataFrame([classification_missing]))

if simplification_missing.get('simple_clause', 0) > 0:
    print('Expected: simple_clause is blank until manual simplifications are added.')
if classification_missing.get('risk_level', 0) > 0:
    raise ValueError('Classification dataset contains missing risk_level labels.')

## 6. Review Class Distribution

In [ ]:
if classification_rows:
    for column in ['clause_type', 'risk_level', 'risk_type']:
        distribution = class_distribution(classification_rows, column)
        print(f'{column} distribution')
        display(pd.DataFrame(distribution.items(), columns=[column, 'count']))
else:
    print('No classification rows to summarize.')

## 7. Review Train, Validation, and Test Splits

For very small datasets, not every split can be populated. With at least three clauses, the builder creates train, validation, and test rows.

In [ ]:
print('Simplification split distribution')
display(pd.DataFrame(split_distribution(simplification_rows).items(), columns=['split', 'count']))

print('Classification split distribution')
display(pd.DataFrame(split_distribution(classification_rows).items(), columns=['split', 'count']))

if len(clauses_df) < 3:
    print('Warning: fewer than 3 clauses. Add more clauses before model training so all splits are useful.')
elif set(simplification_df['split']) != {'train', 'validation', 'test'}:
    print('Warning: expected train, validation, and test splits are not all present.')

## 8. Save Datasets

In [ ]:
SIMPLIFICATION_PATH.parent.mkdir(parents=True, exist_ok=True)
simplification_df.to_csv(SIMPLIFICATION_PATH, index=False)
classification_df.to_csv(CLASSIFICATION_PATH, index=False)

print(f'Saved {len(simplification_df)} row(s) to {SIMPLIFICATION_PATH.relative_to(PROJECT_ROOT)}')
print(f'Saved {len(classification_df)} row(s) to {CLASSIFICATION_PATH.relative_to(PROJECT_ROOT)}')